# ViNumQA — FP16 eval-only: P vs G_all step 20

Đánh giá cùng 32 câu validation, không training thêm, không mở test.

- P được chạy lại với FP16; không thay bằng điểm P BF16 của version 6.
- G_all nạp đúng adapter step 20 từ output version 6, kiểm tra condition và hash weights.
- Nếu session còn working files, tái sử dụng chúng; nếu không, dùng KaggleHub khôi phục output version 6.
- Nếu KaggleHub không mount được nguồn private, Add Input → Notebook Output → notebook này → version 6.
- Bạn tự bật GPU và Run All; bản sửa này được lưu bằng Quick Save, không tự chạy GPU.

Model: [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B), cùng revision đã pin.


In [ ]:
# BẢNG ĐIỀU KHIỂN: chỉnh các cờ ở cell này, rồi chạy lại các cell phụ thuộc.
from pathlib import Path
import tempfile

# Chạy phần tải/audit dữ liệu trên CPU; chưa nạp student hay bắt đầu training.
RUN_SETUP = True
# Chỉ bổ sung thư viện CPU đang thiếu, ví dụ SymPy cho so sánh chương trình.
INSTALL_CPU_DEPS = True
# Bật khi chuẩn bị chạy GPU; thư viện được pin, không cài lại toàn bộ Torch/CUDA.
INSTALL_GPU_STACK = True
# Bật riêng baseline model gốc; kết quả mặc định là smoke trên validation.
RUN_P = True
# Eval-only: tuple rỗng để các block training không được gọi.
TRAIN_ARMS = ()
# Tách bước chấm adapter khỏi training để tránh vô tình chạy lại cả hai.
EVAL_ARMS = ("G_all",)
# So checkpoint bằng full validation EA, không chỉ validation loss.
RUN_CHECKPOINT_SELECTION = False
# Chỉ mở test sau khi khóa phương pháp, checkpoint và registration.
ALLOW_FINAL_TEST = False
EVAL_SPLIT = "valid"
# 32 câu là smoke sample. Đặt None khi cần đủ tập; final test không cho dùng subset.
MAX_EVAL_ITEMS = 32  # smoke/development; None for the full frozen split
# Seed cố định để tái lập các lựa chọn ngẫu nhiên; không chọn seed tốt nhất từ test.
SEED = 42
# Model Hugging Face: https://huggingface.co/Qwen/Qwen3-1.7B
# Các file weights/tokenizer tại bản pin: https://huggingface.co/Qwen/Qwen3-1.7B/tree/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e
MODEL_ID = "Qwen/Qwen3-1.7B"
# Pin commit model/tokenizer để lần chạy sau không vô tình tải weights mới.
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
# Tổng context cho input và output; code báo overflow thay vì cắt input âm thầm.
MAX_LENGTH = 4096
# Trần token sinh chương trình; overflow hoặc output lỗi vẫn nằm trong mẫu số đánh giá.
MAX_NEW_TOKENS = 256
TRAIN_EPOCHS = 2
# Giá trị dương giới hạn optimizer step và ghi đè số epoch; 20 phù hợp smoke đầu tiên.
MAX_TRAIN_STEPS = 20  # set 20 for the first GPU smoke test
REPORT_KEY_VERSION = "path-parent-v1"
METRIC_ID = "finqa-vinumqa-string-rowalias-v1"
# Working disk của Kaggle là tạm thời: cần lưu version/output trước khi kết thúc session.
WORK_DIR = (Path("/kaggle/working/vinumqa-baselines")
            if Path("/kaggle/working").exists()
            else Path(tempfile.gettempdir()) / "vinumqa-baselines-artifacts")
# File JSONL từ teacher: source_id, candidate_id, program; chỉ nhận source thuộc train sạch.
CANDIDATES_FILE = WORK_DIR / "candidates.jsonl"
FINAL_REGISTRATION_FILE = WORK_DIR / "final_registration.json"
# Optional: choose an exported adapter. Otherwise validation uses the last adapter.
# Điền đường dẫn adapter đã lưu nếu mở session mới hoặc muốn chấm lại checkpoint cụ thể.
ADAPTER_PATHS = {}
# Optional: explicit checkpoints to compare on full validation EA before final test.
# Ví dụ {'G_all': ['.../checkpoint-50', '.../checkpoint-100']} để chọn theo EA.
CHECKPOINT_CANDIDATES = {}

# Pin dataset và checksum từng tệp; URL cập nhật nhưng khác nội dung sẽ bị chặn.
DATA_COMMIT = "281bdcd70abde7753d95014dbd9babec7f97f3fd"
DATA_BASE = ("https://raw.githubusercontent.com/ThanhDatVN/"
             f"vinumqa-numerical-reasoning/{DATA_COMMIT}/data/")
DATA_HASHES = {
    "train": "81d44c30d47d96202810f3b4a829382ec110aae6cbd3e9366a4e46253001108e",
    "valid": "07eadf7d185c5d42211c624f3b32402802edf57fe5b6fe64337778a0d5524c4d",
    "test": "8a62a31ac8ab5276054b24fdab89262230078fe4754c817262311c5b30b623db",
}
# Scorer cũng được pin độc lập với dataset để việc chấm không thay đổi giữa các lần chạy.
SCORER_COMMIT = "0f16e2867befa6840783e58be38c9efb9229d742"
SCORER_URL = ("https://raw.githubusercontent.com/czyssrs/FinQA/"
              f"{SCORER_COMMIT}/code/evaluate/evaluate.py")
SCORER_SHA256 = "845cd131cab843eceff256cf6d392978cc470a7da4a80107beb56027fdca5c13"
# Giữ stack GPU nhất quán; environment.json lưu lại phiên bản đã cài thực tế.
GPU_PACKAGES = [
    "transformers==4.57.3", "peft==0.17.1",
    "accelerate==1.11.0", "bitsandbytes==0.48.2", "kagglehub==1.0.3",
]

# Mã định danh triển khai đã kiểm thử; lần bổ sung comment này giữ nguyên executable AST.
NOTEBOOK_CODE_SHA256 = "48ad2d0467a1a22d66a944bc2f76077873cdcc3b4d4282045231deb266a6d5d2"

# Profile eval-only FP16: nạp adapter cũ, không train và không mở test.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
MEASUREMENT_CARD_ID = "MC-107"
MEASUREMENT_CARD_SHA256 = "f3daeeb44d86e95370a91e631254ccc14763faff8ff75a37d6787df96f7fd519"
MEASUREMENT_REGISTRY_SHA256 = "edb42f15393ccc530d5e62de07c750eb997f241cb06e0da1cf106ca2ccfc88cf"
LAUNCH_LABEL = "mc107-vinumqa-fp16-eval32-v1"
WORK_DIR = WORK_DIR / LAUNCH_LABEL
CANDIDATES_FILE = WORK_DIR / "candidates.jsonl"
FINAL_REGISTRATION_FILE = WORK_DIR / "final_registration.json"


# Bắt buộc dùng FP16 trên T4; không nhận BF16 giả lập là hỗ trợ native.
COMPUTE_PRECISION = "float16"
# Nguồn adapter cố định, không dùng latest để tránh đổi weights giữa các lượt.
SOURCE_NOTEBOOK_OUTPUT = "tungnguyen05/vinumqa-baseline-setup-p-g-b-c/versions/6"
SOURCE_TRAINING_PATH = "vinumqa-baselines/mc106-vinumqa-gpu-smoke-v1/training/G_all/seed-42-2bc1a7d7d347"
SOURCE_TRAINING_CONDITION_HASH = "2bc1a7d7d347e1ddad1fb7714d8ddd020121c7e5bff4aa8962bd92677d8aeb95"
SOURCE_ADAPTER_SHA256 = "11a5904761c3d5fc00964262e0b97b311593de4a192592cdb8ce4e829dbded1e"
SOURCE_SPLIT_HASH = "ec29d6e2f6287b17dcbab41e959a0e53efac8e6bd8b42ef9521730c48e2415fd"
CURRENT_EVAL_RUNS = {}


## 1. Môi trường và helper artifact

Chỉ cài dependency CPU còn thiếu. GPU stack là block riêng và chỉ chạy khi bật cờ.
Các phiên bản runtime thật được ghi trong environment.json; không in biến môi trường hay credentials.

In [ ]:
# CHUẨN BỊ MÔI TRƯỜNG: chỉ cài dependency khi cần và ghi lại phiên bản thực tế.
import contextlib
import hashlib
import importlib.metadata
import importlib.util
import io
import json
import math
import os
import platform
import re
import subprocess
import sys
import time
import urllib.request

# Hash JSON ổn định dù thứ tự key khác nhau; dùng để định danh điều kiện chạy.
def canonical_hash(value):
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False).encode()
    return hashlib.sha256(raw).hexdigest()

# Ghi file tạm rồi thay thế đích, giảm nguy cơ để lại JSON dở khi phiên bị ngắt.
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    # Thay thế ở cuối sau khi JSON đã được ghi xong.
    tmp.replace(path)

# Tái dùng cache chỉ khi đúng SHA-256; nguồn khác nội dung phải báo lỗi.
def download_verified(url, path, sha256):
    path = Path(path)
    # Không tải lại dữ liệu khi cache đã tồn tại; vẫn kiểm hash bên dưới.
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        request = urllib.request.Request(url, headers={"User-Agent": "VMLU-notebook-setup"})
        raw = urllib.request.urlopen(request, timeout=60).read()
        if hashlib.sha256(raw).hexdigest() != sha256:
            raise ValueError("Downloaded source hash mismatch")
        temporary = path.with_suffix(path.suffix + ".download")
        temporary.write_bytes(raw)
        temporary.replace(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f"Cached source hash mismatch: {path.name}")
    return path

# Import/cài dependency CPU không đồng nghĩa đã nạp model lên GPU.
if INSTALL_CPU_DEPS:
    missing = [name for name in ("numpy", "sympy", "tqdm")
               if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Cài stack này chỉ khi người dùng bật cờ, để Run All mặc định nhẹ.
if INSTALL_GPU_STACK:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *GPU_PACKAGES], check=True)

WORK_DIR.mkdir(parents=True, exist_ok=True)
# Ghi phiên bản package phục vụ provenance; không đọc hoặc in credentials.
versions = {}
for package in ("numpy", "sympy", "tqdm", "torch", "transformers", "peft", "accelerate", "bitsandbytes", "kagglehub"):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None
write_json(WORK_DIR / "environment.json", {"python": platform.python_version(), "packages": versions})
print(json.dumps({"python": platform.python_version(), "packages": versions,
                  "gpu_work_enabled": bool(RUN_P or TRAIN_ARMS or EVAL_ARMS)}, indent=2))

# Xác nhận GPU thực và ghi artifact trước khi nạp weights.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle did not allocate a CUDA GPU")
gpu_report = {"name": torch.cuda.get_device_name(0),
              "compute_capability": list(torch.cuda.get_device_capability(0)),
              "requested_precision": COMPUTE_PRECISION,
              "visible_devices": torch.cuda.device_count(),
              "torch": str(torch.__version__), "cuda": torch.version.cuda,
              "vram_bytes": torch.cuda.get_device_properties(0).total_memory}
write_json(WORK_DIR / "gpu_report.json", gpu_report)
print("GPU_PREFLIGHT", json.dumps(gpu_report))

write_json(WORK_DIR / "launch_registration.json", {"card_id": MEASUREMENT_CARD_ID, "detail_sha256": MEASUREMENT_CARD_SHA256, "registry_sha256": MEASUREMENT_REGISTRY_SHA256, "notebook_code_sha256": NOTEBOOK_CODE_SHA256, "label": LAUNCH_LABEL})


In [ ]:
# HÀM DỮ LIỆU THUẦN: kiểm tra định danh, tách nguồn và dựng input không lẫn đáp án.
import hashlib
import json
from pathlib import Path

# Gắn fingerprint với prompt thực tế và model/runtime thực tế để resume không lấy nhầm kết quả cũ.
def bind_inference_condition(condition, rows, prompts, model_id, model_revision, runtime, adapter_hash):
    # Một prompt phải ứng với đúng một ID; sai manifest cần dừng ngay.
    if len(rows) != len(prompts):
        raise ValueError("Prompt manifest length mismatch")
    bound = dict(condition)
    bound.update({
        "actual_model_id": model_id, "actual_model_revision": model_revision,
        "actual_tokenizer_revision": model_revision, "inference_runtime": dict(runtime),
        "adapter_sha256": adapter_hash,
        "inference_prompt_sha256": hashlib.sha256(
            json.dumps([{"id": row["id"], "prompt": prompt}
                        for row, prompt in zip(rows, prompts, strict=True)],
                       ensure_ascii=False, sort_keys=True).encode()).hexdigest(),
    })
    return bound

# Adapter LoRA phải được gắn vào đúng base model và revision đã dùng khi train.
def validate_base_identity(trained, model_id, model_revision):
    # Cùng kiến trúc nhưng khác revision vẫn là điều kiện inference khác.
    if trained.get("model_id") != model_id or trained.get("model_revision") != model_revision:
        raise ValueError("Adapter base model differs from the training condition")

# Hash cả config và nội dung weights; cùng đường dẫn chưa chắc là cùng adapter.
def adapter_fingerprint(path):
    path = Path(path)
    files = [path / "adapter_config.json"]
    weights = [path / name for name in ("adapter_model.safetensors", "adapter_model.bin")
               if (path / name).is_file()]
    if not weights or not files[0].is_file():
        raise ValueError("Missing adapter config or weights")
    digest = hashlib.sha256()
    # Đọc theo chunk để hash weights mà không cần nạp model hay giữ cả file trong RAM.
    for file in files + weights:
        digest.update(file.name.encode())
        with file.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()

# Lấy nhóm nguồn từ phần trước tên trang/câu hỏi: cùng báo cáo phải ở cùng split.
def report_key(item_id):
    if not isinstance(item_id, str) or "/" not in item_id:
        raise ValueError("Unsupported source identity")
    return item_id.rsplit("/", 1)[0]

# Ưu tiên test trước, validation sau, rồi loại các nhóm nguồn ấy khỏi train.
def make_clean_splits(raw):
    for split in ("train", "valid", "test"):
        rows = raw[split]
        seen = set()
        for row in rows:
            # ID trùng không được xử lý bằng cách âm thầm lấy một bản.
            if row["id"] in seen:
                raise ValueError(f"duplicate identity in {split}: {row['id']}")
            seen.add(row["id"])
            if not all(key in row for key in ("pre_text", "post_text", "table", "qa")):
                raise ValueError("Invalid source schema")
            if "question" not in row["qa"]:
                raise ValueError("Missing question")
    # Thứ tự ưu tiên này tránh dùng cùng báo cáo để học rồi đánh giá trên câu khác của nó.
    # Kiểm tra lại sau lọc, thay vì chỉ tin vào quy trình tách ở trên.
    test_groups = {report_key(row["id"]) for row in raw["test"]}
    valid = [row for row in raw["valid"] if report_key(row["id"]) not in test_groups]
    valid_groups = {report_key(row["id"]) for row in valid}
    train = [row for row in raw["train"]
             if report_key(row["id"]) not in test_groups | valid_groups]
    clean = {"train": train, "valid": valid, "test": list(raw["test"])}
    groups = {key: {report_key(row["id"]) for row in rows} for key, rows in clean.items()}
    if groups["train"] & groups["valid"] or groups["train"] & groups["test"] or groups["valid"] & groups["test"]:
        raise ValueError("Source overlap after filtering")
    audit = {"raw_counts": {k: len(v) for k, v in raw.items()},
             "counts": {k: len(v) for k, v in clean.items()},
             "groups": {k: len(v) for k, v in groups.items()}}
    return clean, audit

# Chỉ chọn field input cho model; gold program và exe_ans chỉ dành cho target/scorer.
def render_prompt(row):
    # Explicit input whitelist: never serialize qa.program, exe_ans or gold retrieval.
    table = "\n".join(" | ".join(str(cell) for cell in values) for values in row["table"])
    return (
        "Dựa vào văn bản và bảng, hãy trả lời bằng một chương trình FinQA DSL.\n"
        "Dùng add, subtract, multiply, divide, exp, greater hoặc các phép table_*.\n"
        "Tham chiếu bước trước bằng #0, #1; ví dụ: subtract(12, 5), divide(#0, 5).\n"
        "Chỉ xuất một dòng Program: <chương trình>. Không dùng Python hay giải thích.\n\n"
        "Văn bản trước bảng:\n" + "\n".join(row["pre_text"]) +
        "\n\nBảng:\n" + table +
        "\n\nVăn bản sau bảng:\n" + "\n".join(row["post_text"]) +
        "\n\nCâu hỏi:\n" + row["qa"]["question"]
    )

# Lấy cùng thứ tự mẫu tất định cho mọi arm; final test phải dùng đủ mẫu số.
def evaluation_rows(splits, split, allow_test=False, limit=None):
    if split not in ("valid", "test"):
        raise ValueError("Evaluation split must be valid or test")
    # Chặn việc nhìn test sớm khi đang chỉnh phương pháp.
    if split == "test" and not allow_test:
        raise ValueError("Final test is disabled; freeze registration and checkpoint selection first")
    if split == "test" and limit is not None:
        raise ValueError("Final test must use its full denominator")
    # Hash ID tạo thứ tự ổn định; mọi arm smoke dùng cùng tập câu.
    rows = sorted(splits[split], key=lambda row: hashlib.sha256(row["id"].encode()).hexdigest())
    return rows if limit is None else rows[:limit]

# Nối prompt với target, nhưng chỉ cho phần target đóng góp vào training loss.
def make_feature(prompt_ids, target_ids, max_length):
    # None là tín hiệu loại có ghi nhận, không phải target bị cắt bớt.
    if not target_ids or len(prompt_ids) + len(target_ids) > max_length:
        return None
    return {"input_ids": list(prompt_ids) + list(target_ids),
            "attention_mask": [1] * (len(prompt_ids) + len(target_ids)),
            "labels": [-100] * len(prompt_ids) + list(target_ids)}


# Bảo đảm lượt smoke ngắn cũng tạo checkpoint đầy đủ trước khi kết thúc.
def training_save_steps(max_steps):
    return min(50, max_steps) if max_steps > 0 else 50


## 2. Dữ liệu pin và split theo báo cáo

Nguồn là bản chia sẻ của nhóm triển khai; không đổi tên thành bản chính thức của ban tổ chức.
Ưu tiên test > validation > train. Tách theo đường dẫn báo cáo, không chỉ theo ID câu hỏi.
Số dự kiến từ audit trước: 2.038 / 485 / 497; kiểm tra số thật ở output.
Việc nhóm theo đường dẫn vẫn cần kiểm toán alias/tài liệu tương đương trước nghiên cứu xác nhận.

In [ ]:
# TẢI VÀ AUDIT: kiểm hash trước, sau đó mới tách train/validation/test.
if RUN_SETUP:
    raw_splits = {}
    # Lưu bản raw riêng để có thể kiểm toán lại tách split.
    for split, sha256 in DATA_HASHES.items():
        path = download_verified(DATA_BASE + split + ".json",
                                 WORK_DIR / "raw" / f"{split}.json", sha256)
        raw_splits[split] = json.loads(path.read_text(encoding="utf-8"))
    # Không chia ngẫu nhiên theo từng câu, vì nhiều câu có thể chung một báo cáo.
    splits, split_audit = make_clean_splits(raw_splits)
    split_manifest = {
        "protocol_id": "vinumqa-baseline-setup-v1",
        "data_commit": DATA_COMMIT, "data_hashes": DATA_HASHES,
        "report_key_version": REPORT_KEY_VERSION,
        "ids": {k: [row["id"] for row in rows] for k, rows in splits.items()},
        "audit": split_audit,
    }
    # Hash tính trước khi thêm chính trường manifest_hash để tránh tự tham chiếu.
    SPLIT_HASH = canonical_hash(split_manifest)
    split_manifest["manifest_hash"] = SPLIT_HASH
    write_json(WORK_DIR / "split_manifest.json", split_manifest)
    print(json.dumps(split_audit, ensure_ascii=False, indent=2))


## 3. Adapter và bộ chấm EA / PA

Tải nguyên bản evaluate.py của FinQA ở commit pin, kiểm SHA-256 trước import.
Hai quy tắc tương thích được ghi thành metric ID riêng:

1. ViNumQA lưu exe_ans dạng chuỗi: chuyển số/yes/no về kiểu tương ứng và làm tròn số
   5 chữ số như kết quả thực thi upstream.
2. Nhãn hàng bảng có ngoặc như EPS (VND) được mã hoá thành alias ổn định ở cả table,
   gold program và predicted program trước khi gọi upstream. Giá trị bảng không đổi.

Không sửa phép tính hoặc symbolic comparison của upstream. Chương trình lỗi vẫn có EA/PA = 0.
Metric này chưa được tuyên bố tương đương scorer chính thức của shared task.

In [ ]:
# CHẤM ĐIỂM: adapter định dạng ViNumQA, còn phép tính và PA dùng scorer FinQA đã pin.
import contextlib
import io
import math
import re

# Theo contract output, lấy dòng Program cuối cùng; không sửa phép tính của model.
def extract_program(text):
    text = str(text).strip()
    matches = re.findall(r"(?im)^Program:\s*(.+)$", text)
    return matches[-1].strip() if matches else text

# Mã hóa nhãn hàng chứa ngoặc thành alias an toàn cho parser upstream; các giá trị giữ nguyên.
def alias_table_rows(program, table):
    labels = {str(row[0]): f"row_alias_{index}" for index, row in enumerate(table) if row}
    aliased = [[labels[str(row[0])], *row[1:]] if row else row for row in table]
    # Thay nhãn dài trước để hạn chế việc thay một nhãn con trong nhãn dài hơn.
    for label in sorted(labels, key=len, reverse=True):
        for op in ("table_max", "table_min", "table_sum", "table_average"):
            program = program.replace(f"{op}({label}, ", f"{op}({labels[label]}, ")
    return program, aliased

# Đổi gold dạng chuỗi về số hoặc yes/no; làm tròn số 5 chữ số theo kết quả upstream.
def normalized_gold(value, upstream):
    if isinstance(value, bool):
        return "yes" if value else "no"
    text = str(value).strip()
    # Một số câu so sánh có đáp án yes/no, không phải số.
    if text.lower() in ("yes", "no"):
        return text.lower()
    # Tận dụng quy tắc đọc số/phần trăm của scorer, không tự đặt tolerance mới.
    number = upstream.str_to_num(text)
    if isinstance(number, (int, float)) and math.isfinite(number):
        return round(number, 5)
    return None

# Kiểm tra cấu trúc DSL và tên phép toán trước khi giao cho scorer; không thực thi Python tự do.
def checked_tokens(program, upstream):
    tokens = upstream.program_tokenization(program)
    # Mỗi bước DSL gồm 4 token và toàn bộ chương trình kết thúc bằng EOF.
    if not program or len(tokens) < 5 or (len(tokens) - 1) % 4 or len(tokens) > 129:
        raise ValueError("Invalid DSL structure")
    for index in range(0, len(tokens) - 1, 4):
        if tokens[index] not in [op + "(" for op in upstream.all_ops] or tokens[index + 3] != ")":
            raise ValueError("Unsupported DSL")
    return tokens

# EA kiểm kết quả cuối, PA kiểm tương đương chương trình; hai tiêu chí không đồng nghĩa.
def score_program(program, row, upstream):
    result = {"valid": False, "ea": False, "pa": False, "value": None, "error": None}
    try:
        predicted = extract_program(program)
        # Gold và prediction phải dùng cùng ánh xạ nhãn hàng để PA còn so được.
        predicted, table = alias_table_rows(predicted, row["table"])
        gold_program, _ = alias_table_rows(row["qa"]["program"], row["table"])
        pred_tokens = checked_tokens(predicted, upstream)
        with contextlib.redirect_stdout(io.StringIO()):
            invalid, value = upstream.eval_program(pred_tokens, table)
        # Chia 0, số không hữu hạn hoặc chương trình không chạy được đều là đáp án lỗi.
        if invalid or (isinstance(value, (int, float)) and not math.isfinite(value)):
            result["error"] = "execution_invalid"
            return result
        result["valid"] = True
        result["value"] = value
        target = normalized_gold(row["qa"]["exe_ans"], upstream)
        # EA là khớp đáp án cuối sau adapter, chưa chứng minh lời giải đúng cấu trúc.
        result["ea"] = target is not None and value == target
        try:
            gold_tokens = checked_tokens(gold_program, upstream)
            with contextlib.redirect_stdout(io.StringIO()):
                # PA sử dụng symbolic comparison của upstream, không so hai chuỗi bằng nhau.
                result["pa"] = bool(upstream.equal_program(gold_tokens, pred_tokens))
        except Exception:
            result["error"] = "gold_or_symbolic_comparison_invalid"
    # Trả trạng thái lỗi để giữ câu trong mẫu số; không vá output của model.
    except Exception as exc:
        result["error"] = type(exc).__name__
    return result


In [ ]:
# KIỂM TRA SCORER BẰNG GOLD: đây là audit tham chiếu, chưa phải điểm của một model.
if RUN_SETUP:
    scorer_path = download_verified(SCORER_URL, WORK_DIR / "upstream" / "evaluate.py", SCORER_SHA256)
    # Chỉ import file sau khi checksum đã khớp; đây là code upstream đã pin.
    scorer_spec = importlib.util.spec_from_file_location("finqa_pinned", scorer_path)
    upstream = importlib.util.module_from_spec(scorer_spec)
    scorer_spec.loader.exec_module(upstream)
    def scorer(program, row):
        return score_program(program, row, upstream)
    reference_audit = {}
    # No test gold replay during setup.
    # Audit tham chiếu chỉ trên train/validation, không replay gold test ở bước phát triển.
    for split in ("train", "valid"):
        scored = [scorer(row["qa"]["program"], row) for row in splits[split]]
        reference_audit[split] = {
            "n": len(scored),
            "gold_execution_valid": sum(x["valid"] for x in scored),
            "gold_matches_answer": sum(x["ea"] for x in scored),
            "gold_self_pa": sum(x["pa"] for x in scored),
        }
        # Lưu ID và lỗi tham chiếu để người nghiên cứu kiểm tra; không tự sửa gold.
        write_json(WORK_DIR / "audit" / f"{split}_reference_failures.json",
                   [{"id": row["id"], "program": row["qa"]["program"], "score": result}
                    for row, result in zip(splits[split], scored, strict=True)
                    if not (result["valid"] and result["ea"] and result["pa"])])
    write_json(WORK_DIR / "reference_audit.json",
               {"metric_id": METRIC_ID, "scorer_hash": SCORER_SHA256, "splits": reference_audit})
    print(json.dumps(reference_audit, indent=2))
    print("Reference failures are audit findings; no test label is repaired.")


## 4. Tạo các tập G_all / G / B / C

G_all dùng gold hợp lệ của train sạch và có thể chuẩn bị ngay.
File candidates.jsonl (nếu có) chứa source_id, candidate_id, program.
Nên lưu thêm teacher_model, prompt_hash, usage, condition_hash và raw_response để có provenance.
Chỉ chấp nhận ID thuộc train sạch; điểm EA/PA do candidate tự khai được tính lại bằng scorer pin.
B và C chọn từ cùng pool; G lấy gold trên đúng giao nguồn đó.

In [ ]:
# TẠO TARGET HUẤN LUYỆN: G_all dùng gold; G/B/C dùng cùng giao nguồn để đối chiếu.
import hashlib

# Dựng các arm từ cùng train split; luôn tính lại EA/PA, không tin điểm ghi sẵn trong candidate.
def make_training_packs(train_rows, candidates, score, seed=42):
    by_id = {row["id"]: row for row in train_rows}
    if len(by_id) != len(train_rows):
        raise ValueError("duplicate training identity")
    packs = {"G_all": [], "G": [], "B": [], "C": []}
    for row in train_rows:
        # G_all chỉ lấy gold thực thi được, khớp đáp án và có PA hợp lệ.
        ref = score(row["qa"]["program"], row)
        if ref["valid"] and ref["ea"] and ref["pa"]:
            packs["G_all"].append({"id": row["id"], "prompt": render_prompt(row),
                                  "target": "Program: " + row["qa"]["program"]})
    eligible_gold = {row["id"] for row in packs["G_all"]}
    buckets, seen = {}, set()
    for candidate in candidates:
        source_id = candidate["source_id"]
        # Không cho candidate từ validation/test đi vào training.
        if source_id not in by_id:
            raise ValueError("candidate source is outside clean training split")
        identity = (source_id, candidate["candidate_id"])
        if identity in seen:
            raise ValueError("duplicate candidate identity")
        seen.add(identity)
        # EA/PA được tính lại tại đây; field điểm do teacher khai không được dùng.
        result = score(candidate["program"], by_id[source_id])
        buckets.setdefault(source_id, []).append((candidate, result))
    for source_id in sorted(buckets):
        if source_id not in eligible_gold:
            continue
        choices = buckets[source_id]
        # B chỉ yêu cầu kết quả đúng; C còn yêu cầu tương đương chương trình gold.
        b = [c for c, r in choices if r["valid"] and r["ea"]]
        c = [c for c, r in choices if r["valid"] and r["ea"] and r["pa"]]
        # Chỉ giữ giao nguồn có target cho cả hai arm để không lẫn hiệu ứng chọn câu dễ/khó.
        if not b or not c:
            continue
        # Chọn candidate tất định bằng hash; không dùng validation/test để chọn lời giải có lợi.
        def key(candidate):
            return hashlib.sha256(f"{seed}:{source_id}:{candidate['candidate_id']}".encode()).hexdigest()
        b, c = min(b, key=key), min(c, key=key)
        source = by_id[source_id]
        # G lấy chương trình gold, B/C lấy candidate; input và schema output đều giống nhau.
        for arm, program in (("G", source["qa"]["program"]),
                             ("B", b["program"]), ("C", c["program"])):
            packs[arm].append({"id": source_id, "prompt": render_prompt(source),
                               "target": "Program: " + extract_program(program)})
    if not (set(x["id"] for x in packs["G"]) == set(x["id"] for x in packs["B"])
            == set(x["id"] for x in packs["C"])):
        raise ValueError("Matched source IDs differ")
    # Tỷ lệ B/C khác target cho biết bộ lọc có tạo ra can thiệp đáng kể hay không.
    audit = {"matched_sources": len(packs["G"]), "gold_all_sources": len(packs["G_all"]),
             "candidate_count": len(candidates),
             "different_B_C_targets": sum(b["target"] != c["target"]
                                          for b, c in zip(packs["B"], packs["C"], strict=True))}
    return packs, audit


In [ ]:
# XUẤT DỮ LIỆU: chỉ đọc teacher pool có sẵn, không gọi API teacher trong cell này.
if RUN_SETUP:
    candidates = []
    # Chưa có file thì chỉ chuẩn bị G_all; không tạo candidate giả để lấp chỗ trống.
    if CANDIDATES_FILE.exists():
        candidates = [json.loads(line) for line in CANDIDATES_FILE.read_text().splitlines() if line.strip()]
    packs, pack_audit = make_training_packs(splits["train"], candidates, scorer, SEED)
    # JSONL này chứa target để train; render_prompt vẫn tách gold khỏi input.
    for arm, records in packs.items():
        path = WORK_DIR / "exports" / f"{arm}.jsonl"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records), encoding="utf-8")
    write_json(WORK_DIR / "pack_audit.json", pack_audit)
    print(json.dumps(pack_audit, indent=2))
    if not candidates:
        print("G_all ready. G/B/C matched wait for a real teacher pool; no teacher request was sent.")


## Khôi phục adapter version 6 trước khi nạp model

Kiểm tra nguồn, base model, split, step 20 và hash weights; không chấp nhận adapter khác chỉ vì trùng tên.


In [ ]:

# Precision được chọn theo năng lực phần cứng native, không tính BF16 giả lập.
def select_compute_precision(requested, capability):
    if requested not in ("float16", "bfloat16", "auto"):
        raise ValueError("Unknown compute precision")
    native_bf16 = capability[0] >= 8
    if requested == "float16":
        return "float16"
    if requested == "bfloat16" and not native_bf16:
        raise ValueError("GPU does not support native BF16; use float16")
    return "bfloat16" if native_bf16 else "float16"


# Kiểm tra metadata và bytes trước khi đưa adapter vào PEFT.
def validate_source_adapter(path, expected):
    path = Path(path)
    parent = path.parent
    condition = json.loads((parent / "condition.json").read_text())
    complete = json.loads((parent / "training_complete.json").read_text())
    condition_hash = canonical_hash(condition)
    if condition_hash != expected["condition_hash"] or complete.get("condition_hash") != condition_hash:
        raise ValueError("Source training condition hash mismatch")
    if condition.get("arm") != "G_all":
        raise ValueError("Source training arm mismatch")
    if condition.get("split_hash") != expected["split_hash"]:
        raise ValueError("Source adapter split mismatch")
    validate_base_identity(condition, expected["model_id"], expected["model_revision"])
    if complete.get("optimizer_step") != expected["optimizer_step"]:
        raise ValueError("Source adapter optimizer step mismatch")
    if adapter_fingerprint(path) != expected["adapter_sha256"]:
        raise ValueError("Source adapter content hash mismatch")
    return {"path": str(path.resolve()), "condition_hash": condition_hash,
            "optimizer_step": complete["optimizer_step"], "training_condition": condition,
            "adapter_sha256": expected["adapter_sha256"]}


def resolve_source_adapter(direct_paths, search_roots, handle, expected, download_fn=None):
    # Cache có sẵn nhưng hỏng phải báo lỗi, không âm thầm thay bằng weights khác.
    for path in direct_paths:
        if Path(path).exists():
            result = validate_source_adapter(path, expected)
            result["restore_method"] = "existing_working_copy"
            return result

    def find_verified(roots):
        for root in roots:
            root = Path(root)
            if not root.exists():
                continue
            candidates = [root] if root.name == "adapter" else sorted(
                p.parent for p in root.rglob("adapter_config.json") if p.parent.name == "adapter")
            for candidate in candidates:
                try:
                    return validate_source_adapter(candidate, expected)
                except (OSError, ValueError, KeyError):
                    continue
        return None

    result = find_verified(search_roots)
    if result is not None:
        result["restore_method"] = "mounted_version6_output"
        return result
    try:
        if download_fn is None:
            import kagglehub
            download_fn = kagglehub.notebook_output_download
        # KaggleHub dùng auth tích hợp trong Kaggle; không nhúng token trong notebook.
        downloaded_root = download_fn(handle)
        result = find_verified([downloaded_root])
    except Exception as exc:
        raise RuntimeError(
            "Cannot restore version 6 automatically. Add Input > Notebook Output > "
            "tungnguyen05/vinumqa-baseline-setup-p-g-b-c, version 6, then rerun setup. "
            "Do not enable training.") from exc
    if result is None:
        raise ValueError("Pinned version 6 output has no matching verified adapter")
    result["restore_method"] = "kagglehub_pinned_version6"
    return result


In [ ]:

# Khôi phục adapter trước khi nạp base model: thiếu weights thì dừng sớm, tránh tốn GPU.
if SPLIT_HASH != SOURCE_SPLIT_HASH:
    raise ValueError("Current dataset manifest differs from version 6")
source_expected = {
    "condition_hash": SOURCE_TRAINING_CONDITION_HASH,
    "adapter_sha256": SOURCE_ADAPTER_SHA256,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "split_hash": SPLIT_HASH,
    "optimizer_step": 20,
}
if ADAPTER_PATHS.get("G_all"):
    SOURCE_ADAPTER = validate_source_adapter(ADAPTER_PATHS["G_all"], source_expected)
    SOURCE_ADAPTER["restore_method"] = "explicit_verified_path"
else:
    SOURCE_ADAPTER = resolve_source_adapter(
        [Path("/kaggle/working") / SOURCE_TRAINING_PATH / "adapter"],
        [Path("/kaggle/input")], SOURCE_NOTEBOOK_OUTPUT, source_expected)
ADAPTER_PATHS["G_all"] = SOURCE_ADAPTER["path"]
write_json(WORK_DIR / "source_adapter.json",
           {key: value for key, value in SOURCE_ADAPTER.items() if key != "training_condition"})
print({"source_version": 6, "source_step": SOURCE_ADAPTER["optimizer_step"],
       "restore_method": SOURCE_ADAPTER["restore_method"], "adapter": SOURCE_ADAPTER["path"]})


## 5. Hàm dùng chung: student, feature và inference

GPU stack chỉ được import khi thực sự gọi các hàm này.
Qwen3 dùng non-thinking ở cả train và inference, cùng model/tokenizer revision.
Input quá dài không bị cắt âm thầm: train lưu danh sách loại; evaluation giữ failure trong mẫu số.

### Model được tải ở đâu và khi nào?

- Repository weights: [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B).
- Bản weights cố định: [revision 70d244cc86cc](https://huggingface.co/Qwen/Qwen3-1.7B/tree/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e).
- Trong hàm load_student: AutoTokenizer.from_pretrained tải tokenizer; AutoModelForCausalLM.from_pretrained tải weights. Repo ID trong MODEL_ID được thư viện chuyển thành URL Hugging Face Hub. File đã có trong cache sẽ được tái sử dụng.
- Chạy cell định nghĩa hàm chưa tải model. Model chỉ được nạp khi bật RUN_P, một arm trong TRAIN_ARMS/EVAL_ARMS, hoặc block chọn checkpoint.
- INSTALL_GPU_STACK chỉ cài thư viện; không tải weights. Khi chuẩn bị GPU smoke, bật GPU trong Settings, cài stack rồi đặt RUN_P=True.
- Backbone được nạp ở 4-bit; các lần SFT học thêm adapter LoRA.


Profile này ép FP16 trên T4. BF16 chỉ được phép khi hardware hỗ trợ native; không dùng phép dò gồm emulation.


In [ ]:
# HÀM GPU DÙNG CHUNG: chỉ nạp model khi một block baseline/training thực sự gọi.
# Báo lỗi rõ nếu bật block model nhưng notebook chưa được cấp CUDA GPU.
def require_gpu():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Enable GPU in Kaggle Settings before enabling model/training flags")
    return torch

# Nạp cùng backbone/revision cho mọi arm; train thêm LoRA hoặc nạp adapter để inference.
def load_student(seed, adapter_path=None, for_training=False):
    torch = require_gpu()
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
    set_seed(seed)
    # MODEL_ID là repo Hugging Face; from_pretrained tự tải tokenizer nếu cache chưa có.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Chọn precision theo phần cứng thực được cấp, vì không phải GPU nào cũng hỗ trợ BF16.
    precision = select_compute_precision(COMPUTE_PRECISION, torch.cuda.get_device_capability(0))
    dtype = torch.bfloat16 if precision == "bfloat16" else torch.float16
    # Backbone 4-bit giảm VRAM; phần học thêm là các ma trận LoRA nhỏ.
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                     bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    # Đây là lệnh tải weights từ Hugging Face Hub, sau đó nạp backbone 4-bit lên GPU.
    # Hàm này không gọi Ollama; định nghĩa hàm chưa tải, phải có block baseline/training gọi nó.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, quantization_config=quantization,
        device_map={"": 0}, torch_dtype=dtype)
    if for_training:
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        # Chuẩn bị gradient/checkpointing trước khi gắn các tham số adapter.
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        model = get_peft_model(model, LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
            bias="none", task_type="CAUSAL_LM"))
        # Tắt KV cache trong training để phối hợp với gradient checkpointing.
        model.config.use_cache = False
    # Đây là nạp adapter cho inference; resume training đầy đủ nằm trong train_arm.
    elif adapter_path:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
    return model, tokenizer, dtype

# Dùng chat template chính thức với thinking tắt để thống nhất input và ngân sách sinh.
def chat_prefix(tokenizer, prompt):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False,
        add_generation_prompt=True, enable_thinking=False)

# Tokenize prompt và target riêng để mask đúng; trả lại ID bị loại do quá context.
def encode_records(tokenizer, records):
    features, excluded = [], []
    for row in records:
        prefix = tokenizer(chat_prefix(tokenizer, row["prompt"]), add_special_tokens=False)["input_ids"]
        # EOS cho model biết chương trình đã kết thúc; EOS cũng thuộc phần target.
        target = tokenizer(row["target"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        # Prompt có label -100 nên không bị ép học lại như đáp án.
        feature = make_feature(prefix, target, MAX_LENGTH)
        # Các arm có thể overflow khác nhau: trước so sánh chính cần khớp lại ID sau tokenization.
        if feature is None:
            excluded.append(row["id"])
        else:
            features.append(feature)
    return features, excluded

# Định danh arm, dữ liệu và hyperparameter để checkpoint không bị trộn giữa các thí nghiệm.
def condition_payload(arm, records, seed):
    return {"phase": "development", "card_id": MEASUREMENT_CARD_ID,
            "measurement_card_sha256": MEASUREMENT_CARD_SHA256,
            "measurement_card_hash": MEASUREMENT_REGISTRY_SHA256,
            "launch_label": LAUNCH_LABEL, "arm": arm, "seed": seed, "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION, "split_hash": SPLIT_HASH,
            "notebook_code_sha256": NOTEBOOK_CODE_SHA256,
            "metric_id": METRIC_ID, "upstream_scorer_sha256": SCORER_SHA256,
            "targets_hash": canonical_hash(records), "max_length": MAX_LENGTH,
            "epochs": TRAIN_EPOCHS, "max_steps": MAX_TRAIN_STEPS,
            "checkpoint_save_steps": training_save_steps(MAX_TRAIN_STEPS),
            "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "target_modules": "all-linear"},
            "learning_rate": 2e-4, "batch": 1, "gradient_accumulation": 8,
            "thinking": False, "max_new_tokens": MAX_NEW_TOKENS,
            "requested_precision": COMPUTE_PRECISION}

# Thu thập prompt sau chat template, runtime GPU và hash adapter thật trước inference.
def bind_student_condition(condition, model, tokenizer, rows, adapter_path=None):
    torch = require_gpu()
    # Fingerprint dùng prompt thực sau chat template, không chỉ ID của câu hỏi.
    prompts = [chat_prefix(tokenizer, render_prompt(row)) for row in rows]
    runtime = {
        "gpu": torch.cuda.get_device_name(0), "torch": str(torch.__version__),
        "cuda": torch.version.cuda, "model_dtype": str(model.dtype),
        "tokenizer_class": type(tokenizer).__name__,
        "packages": {name: importlib.metadata.version(name)
                     for name in ("transformers", "peft", "accelerate", "bitsandbytes")},
    }
    adapter_hash = (adapter_fingerprint(adapter_path) if adapter_path is not None
                    else condition.get("adapter_sha256"))
    bound = bind_inference_condition(condition, rows, prompts, MODEL_ID, MODEL_REVISION,
                                     runtime, adapter_hash)
    expected_precision = select_compute_precision(COMPUTE_PRECISION, torch.cuda.get_device_capability(0))
    if str(model.dtype) != "torch." + expected_precision:
        raise ValueError("Actual inference dtype differs from the requested precision")
    bound["requested_precision"] = COMPUTE_PRECISION
    bound["inference_seed"] = SEED
    bound["notebook_code_sha256"] = NOTEBOOK_CODE_SHA256
    return bound

# Dọn cache khi kết thúc arm; caller vẫn cần bỏ tham chiếu model bằng del khi thích hợp.
def release_student(model):
    import gc
    import torch
    model.cpu() if not getattr(model, "is_loaded_in_4bit", False) else None
    gc.collect()
    torch.cuda.empty_cache()

# Ghi từng response hoàn tất ra JSONL để có thể tiếp tục khi session bị ngắt.
def predict_rows(model, tokenizer, rows, path, condition):
    torch = require_gpu()
    # Kiểm tra lại trước resume để không dùng cache từ prompt/model/runtime khác.
    if bind_student_condition(condition, model, tokenizer, rows) != condition:
        raise ValueError("Actual inference inputs/model/runtime differ from the saved condition")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fingerprint = canonical_hash(condition)
    completed = {}
    # Response hoàn tất là terminal; resume chỉ bổ sung các ID còn thiếu.
    if path.exists():
        for line in path.read_text().splitlines():
            record = json.loads(line)
            if record["condition_hash"] != fingerprint:
                raise ValueError("Prediction condition changed: choose a new output directory")
            if record["id"] in completed:
                raise ValueError("Duplicate completed prediction")
            completed[record["id"]] = record
    expected = {row["id"] for row in rows}
    if set(completed) - expected:
        raise ValueError("Prediction IDs do not match this evaluation manifest")
    model.eval()
    with path.open("a", encoding="utf-8") as stream:
        for row in rows:
            if row["id"] in completed:
                continue
            encoded = tokenizer(chat_prefix(tokenizer, render_prompt(row)), return_tensors="pt",
                                add_special_tokens=False)
            record = {"id": row["id"], "condition_hash": fingerprint, "raw_response": "",
                      "error": None, "prompt_tokens": int(encoded["input_ids"].shape[1])}
            start = time.monotonic()
            # Chừa trần output trong context; không cắt mất bằng chứng đầu vào.
            if record["prompt_tokens"] + MAX_NEW_TOKENS > MAX_LENGTH:
                record["error"] = "context_overflow"
            else:
                inputs = {key: value.to(model.device) for key, value in encoded.items()}
                # Inference không cần đồ thị gradient; giảm bộ nhớ và giữ model ở eval mode.
                with torch.inference_mode():
                    output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                            pad_token_id=tokenizer.pad_token_id)
                # Bỏ token của prompt trước khi decode để chỉ chấm câu trả lời model sinh.
                generated = output[0, encoded["input_ids"].shape[1]:]
                record["raw_response"] = tokenizer.decode(generated, skip_special_tokens=True)
                record["generated_tokens"] = int(generated.shape[0])
            record["seconds"] = time.monotonic() - start
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")
            stream.flush()
            # Đẩy từng record xuống file để giảm mất tiến độ nếu session bị thu hồi.
            os.fsync(stream.fileno())
            completed[row["id"]] = record
    if set(completed) != expected:
        raise ValueError("Incomplete prediction coverage")
    return [completed[row["id"]] for row in rows]

# Chấm từng ID rồi chia trên đủ số câu đã chọn; response lỗi không bị loại để tăng điểm.
def summarize_predictions(predictions, rows):
    by_id = {row["id"]: row for row in rows}
    scores = []
    for prediction in predictions:
        score = scorer(prediction["raw_response"], by_id[prediction["id"]])
        scores.append({"id": prediction["id"], **score})
    # Mẫu số gồm tất cả câu đã chọn, kể cả response rỗng/overflow/DSL lỗi.
    n = len(rows)
    if not n or len(scores) != n or len({x["id"] for x in scores}) != n:
        raise ValueError("Wrong score denominator")
    return {"n": n, "ea": sum(x["ea"] for x in scores) / n,
            "pa": sum(x["pa"] for x in scores) / n,
            "valid_rate": sum(x["valid"] for x in scores) / n,
            "scores": scores, "metric_id": METRIC_ID}


## 6. Baseline P — model gốc

Mặc định không chạy. Bật RUN_P và GPU để chạy development trên validation.
MAX_EVAL_ITEMS = 32 chỉ là smoke sample; không báo nó như điểm toàn tập.

In [ ]:
# BASELINE P: đo model gốc với prompt sinh chương trình, trước khi SFT.
if RUN_P:
    rows = evaluation_rows(splits, EVAL_SPLIT, ALLOW_FINAL_TEST, MAX_EVAL_ITEMS)
    if EVAL_SPLIT == "test":
        raise ValueError("Use the final registered evaluation block for test")
    model, tokenizer, _ = load_student(SEED)
    condition = condition_payload("P", [], SEED)
    condition["phase"] = "development_fp16_eval32"
    condition.update({"eval_split": EVAL_SPLIT, "eval_ids": [row["id"] for row in rows]})
    # Đóng băng input thực tế trước khi xác định thư mục kết quả.
    condition = bind_student_condition(condition, model, tokenizer, rows)
    out = WORK_DIR / "runs" / "P" / canonical_hash(condition)[:12]
    # Mỗi thư mục mang một condition hash, giúp đối chiếu lại provenance của điểm.
    write_json(out / "condition.json", condition)
    predictions = predict_rows(model, tokenizer, rows, out / "predictions.jsonl", condition)
    report = summarize_predictions(predictions, rows)
    write_json(out / "report.json", report)
    CURRENT_EVAL_RUNS["P"] = {"report": report, "condition": condition, "out": str(out)}
    print({key: value for key, value in report.items() if key != "scores"})
    release_student(model)
    del model
else:
    print("P configured; RUN_P=False.")


## 7. Hàm QLoRA dùng chung và resume

Trainer lưu đầy đủ optimizer/scheduler/RNG ở checkpoint-*. Khôi phục bằng resume_from_checkpoint,
không chỉ load adapter. Thay condition sẽ tạo thư mục mới và không trộn checkpoint.
Block này ghi validation loss để chẩn đoán; adapter cuối không được gọi là best theo EA.
Dùng block chọn checkpoint bên dưới để chọn bằng full validation EA trước final test.

In [ ]:
# HÀM TRAIN CHUNG: các arm dùng cùng quy trình QLoRA, khác bộ target đầu vào.
# Cùng một hàm train cho G_all/G/B/C giúp hạn chế khác biệt ngoài loại supervision.
def train_arm(arm, records, seed):
    # G/B/C chưa có teacher pool sẽ dừng rõ ràng, không train trên tập rỗng.
    if not records:
        raise ValueError(f"{arm} has no training records; provide a teacher pool for matched G/B/C")
    model, tokenizer, dtype = load_student(seed, for_training=True)
    from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
    from transformers.trainer_utils import get_last_checkpoint
    features, excluded = encode_records(tokenizer, records)
    if not features:
        raise ValueError("No training record fits the context budget")
    # Validation loss chỉ dùng gold tham chiếu dùng được; EA về sau vẫn chấm đủ validation.
    validation_records = [
        {"id": row["id"], "prompt": render_prompt(row),
         "target": "Program: " + row["qa"]["program"]}
        for row in splits["valid"] if scorer(row["qa"]["program"], row)["ea"]
    ]
    validation_features, valid_excluded = encode_records(tokenizer, validation_records)
    if not validation_features:
        raise ValueError("No validation feature fits")
    condition = condition_payload(arm, records, seed)
    condition["excluded_training_ids"] = excluded
    condition["excluded_validation_loss_ids"] = valid_excluded
    # Phiên bản thư viện là một phần điều kiện để resume không trộn môi trường khác.
    condition["runtime_packages"] = {
        package: importlib.metadata.version(package)
        for package in ("torch", "transformers", "peft", "accelerate", "bitsandbytes")}
    condition_hash = canonical_hash(condition)
    out = WORK_DIR / "training" / arm / f"seed-{seed}-{condition_hash[:12]}"
    out.mkdir(parents=True, exist_ok=True)
    condition_file = out / "condition.json"
    if condition_file.exists() and json.loads(condition_file.read_text()) != condition:
        raise ValueError("Training condition changed")
    write_json(condition_file, condition)
    # Một run đã hoàn tất được tái sử dụng thay vì âm thầm train thêm lần nữa.
    if (out / "training_complete.json").exists():
        print(f"{arm}: already complete; reusing the terminal run")
        release_student(model)
        return out / "adapter"
    torch = require_gpu()
    # Các arm dùng cùng hyperparameter; chỉ số optimizer step đã tính gradient accumulation.
    args = TrainingArguments(
        output_dir=str(out), num_train_epochs=TRAIN_EPOCHS, max_steps=MAX_TRAIN_STEPS,
        learning_rate=2e-4, warmup_ratio=0.03, per_device_train_batch_size=1,
        per_device_eval_batch_size=1, gradient_accumulation_steps=8,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16),
        optim="adamw_torch", eval_strategy="epoch", save_strategy="steps", save_steps=training_save_steps(MAX_TRAIN_STEPS),
        logging_steps=10, report_to=[], seed=seed, data_seed=seed, remove_unused_columns=False)
    trainer = Trainer(
        model=model, args=args, train_dataset=features, eval_dataset=validation_features,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
    # Checkpoint-* chứa optimizer/scheduler/RNG; adapter riêng không đủ khôi phục đầy đủ.
    last = get_last_checkpoint(str(out))
    started = time.monotonic()
    # Resume chỉ khi cấu hình và thư mục condition không đổi.
    trainer.train(resume_from_checkpoint=last)
    # Lưu adapter cuối để inference; đây chưa phải checkpoint tốt nhất theo validation EA.
    trainer.save_model(str(out / "adapter"))
    tokenizer.save_pretrained(str(out / "adapter"))
    # trainer_state.json là tiến độ Trainer, không thay thế optimizer checkpoint đã lưu theo save_steps.
    trainer.save_state()
    # Thời gian ở đây là của session này; khi tính tổng chi phí cần cộng các session resume.
    write_json(out / "training_complete.json",
               {"condition_hash": condition_hash, "optimizer_step": trainer.state.global_step,
                "seconds_this_session": time.monotonic() - started,
                "train_features": len(features), "train_excluded": len(excluded),
                "selection": "last_adapter_not_validation_ea_selected"})
    print({"arm": arm, "adapter": str(out / "adapter"), "train_features": len(features)})
    del trainer
    release_student(model)
    return out / "adapter"


## 8. Baseline G_all — SFT chương trình gold

Đây là baseline chạy được ngay sau setup, không cần teacher.
Các gold không được adapter thực thi/chấm đúng bị ghi trong audit và không được dùng làm target.
TRAIN_ARMS = ("G_all",) để chạy; dùng MAX_TRAIN_STEPS = 20 cho smoke đầu tiên.

**Profile hiện tại tắt training: TRAIN_ARMS=(). G_all sẽ được nạp từ output version 6 để đánh giá.**


In [ ]:
# BASELINE G_all: điểm khởi đầu SFT bằng các chương trình gold qua audit.
# G_all dùng toàn bộ gold train hợp lệ; G matched chỉ dùng giao nguồn B/C.
if "G_all" in TRAIN_ARMS:
    ADAPTER_PATHS["G_all"] = str(train_arm("G_all", packs["G_all"], SEED))
else:
    print("G_all configured; training disabled.")


## 9. Baselines G / B / C matched

Các block dùng cùng train_arm; chỉ target thay đổi. G không dùng thêm nguồn ngoài giao B/C.
Không tự sinh hoặc bịa candidate để làm cho các baseline trông đã sẵn sàng.
Trước thí nghiệm chính cần kiểm tra context exclusion và số token của ba arm:
nếu nguồn sau tokenization khác nhau, rebuild cùng giao trước khi so sánh.

In [ ]:
# ĐỐI CHỨNG MATCHED: chạy từng arm tuần tự để tránh giữ nhiều model trên GPU.
# Giữ cùng train_arm cho mọi arm để loại supervision là khác biệt chính.
for arm in ("G", "B", "C"):
    if arm in TRAIN_ARMS:
        ADAPTER_PATHS[arm] = str(train_arm(arm, packs[arm], SEED))
    else:
        print(f"{arm}: {len(packs[arm])} matched records; training disabled.")
# Tên arm gõ sai phải báo lỗi, không bị bỏ qua khiến người dùng tưởng đã train.
unknown_arms = set(TRAIN_ARMS) - {"G_all", "G", "B", "C"}
if unknown_arms:
    raise ValueError(f"Unknown training arms: {sorted(unknown_arms)}")


## 10. Chọn checkpoint bằng validation EA

Điền CHECKPOINT_CANDIDATES, bật RUN_CHECKPOINT_SELECTION. So tất cả checkpoint đã chỉ định
trên đủ validation, không dùng test; hòa EA chọn step sớm hơn.
Sau khi so sánh, selection.json ghi checkpoint và manifest đã dùng.

In [ ]:
# CHỌN CHECKPOINT: dùng toàn bộ validation EA; test không tham gia lựa chọn.
if RUN_CHECKPOINT_SELECTION:
    # limit=None buộc dùng đầy đủ validation thay vì smoke sample 32 câu.
    validation = evaluation_rows(splits, "valid", False, None)
    for arm, checkpoint_paths in CHECKPOINT_CANDIDATES.items():
        if not checkpoint_paths:
            raise ValueError("No checkpoint candidate")
        evaluations = []
        for checkpoint in checkpoint_paths:
            checkpoint = Path(checkpoint)
            training_condition = json.loads((checkpoint.parent / "condition.json").read_text())
            if training_condition["arm"] != arm or training_condition["split_hash"] != SPLIT_HASH:
                raise ValueError("Checkpoint belongs to another arm/split")
            # Không gắn checkpoint lên revision backbone khác với lúc training.
            validate_base_identity(training_condition, MODEL_ID, MODEL_REVISION)
            model, tokenizer, _ = load_student(SEED, adapter_path=checkpoint)
            condition = {"phase": "validation_selection", "arm": arm,
                         "checkpoint": str(checkpoint), "training_condition_hash": canonical_hash(training_condition),
                         "split_hash": SPLIT_HASH, "metric_id": METRIC_ID,
                         "model_revision": MODEL_REVISION, "max_length": MAX_LENGTH,
                         "max_new_tokens": MAX_NEW_TOKENS, "thinking": False,
                         "eval_ids": [row["id"] for row in validation]}
            condition = bind_student_condition(condition, model, tokenizer, validation, checkpoint)
            out = WORK_DIR / "selection" / arm / canonical_hash(condition)[:12]
            predictions = predict_rows(model, tokenizer, validation, out / "predictions.jsonl", condition)
            report = summarize_predictions(predictions, validation)
            write_json(out / "report.json", report)
            step = int(checkpoint.name.split("-")[-1])
            # Lưu cả hash weights để việc đổi file ở cùng đường dẫn cũng bị phát hiện.
            evaluations.append({"path": str(checkpoint), "step": step, "ea": report["ea"],
                                "adapter_sha256": condition["adapter_sha256"],
                                "training_condition_hash": canonical_hash(training_condition)})
            release_student(model)
            del model
        # Ưu tiên EA cao nhất; nếu hòa, lấy step sớm hơn để quy tắc lựa chọn tất định.
        best = min(evaluations, key=lambda row: (-row["ea"], row["step"]))
        selection = {"arm": arm, "split_hash": SPLIT_HASH, "metric_id": METRIC_ID,
                     "validation_n": len(validation), "selected": best, "candidates": evaluations}
        write_json(WORK_DIR / "selection" / arm / "selection.json", selection)
        ADAPTER_PATHS[arm] = best["path"]
        print({"arm": arm, "selected_step": best["step"], "validation_ea": best["ea"]})
else:
    print("Checkpoint EA selection configured; disabled.")


## 11. Đánh giá adapter trên validation hoặc final test đã đăng ký

Mặc định EVAL_ARMS rỗng. Validation 32 câu là smoke, không phải kết quả xác nhận.
Final test yêu cầu ALLOW_FINAL_TEST=True, MAX_EVAL_ITEMS=None, selection.json và
final_registration.json khớp split_hash, metric_id, danh sách arms, selected paths,
selected_adapter_hashes và inference_condition_hashes (tính từ input/config trước khi chạy test).
Đăng ký/card của thí nghiệm chính được làm riêng sau pilot; notebook không tự cấp số MC.

In [ ]:
# ĐÁNH GIÁ ADAPTER: validation dùng để phát triển; final test cần điều kiện đã khóa.
if EVAL_ARMS:
    rows = evaluation_rows(splits, EVAL_SPLIT, ALLOW_FINAL_TEST, MAX_EVAL_ITEMS)
    # Chỉ nhánh final test mới đòi registration và checkpoint selection đã khóa.
    if EVAL_SPLIT == "test":
        if not FINAL_REGISTRATION_FILE.exists():
            raise ValueError("Missing final_registration.json")
        registration = json.loads(FINAL_REGISTRATION_FILE.read_text())
        if registration.get("split_hash") != SPLIT_HASH or registration.get("metric_id") != METRIC_ID:
            raise ValueError("Final registration does not match data/scorer")
        # Danh sách arm phải khớp đăng ký, tránh chỉ báo cáo các arm có lợi.
        if set(registration.get("arms", [])) != set(EVAL_ARMS) or not registration.get("card_id"):
            raise ValueError("Final arms/card do not match registration")
        for arm in EVAL_ARMS:
            selection = json.loads((WORK_DIR / "selection" / arm / "selection.json").read_text())
            if (selection["split_hash"] != SPLIT_HASH or selection["metric_id"] != METRIC_ID
                    or selection["selected"]["path"] != str(ADAPTER_PATHS[arm])
                    or registration.get("selected_paths", {}).get(arm) != str(ADAPTER_PATHS[arm])):
                raise ValueError("Selected adapter differs from final registration")
            # Đường dẫn giống nhau chưa đủ: weights sau selection phải còn nguyên.
            current_hash = adapter_fingerprint(ADAPTER_PATHS[arm])
            if (selection["selected"].get("adapter_sha256") != current_hash
                    or registration.get("selected_adapter_hashes", {}).get(arm) != current_hash):
                raise ValueError("Adapter weights changed after checkpoint selection/registration")
    for arm in EVAL_ARMS:
        if arm not in ADAPTER_PATHS:
            raise ValueError(f"Set ADAPTER_PATHS[{arm!r}] before evaluation")
        adapter = Path(ADAPTER_PATHS[arm])
        trained = json.loads((adapter.parent / "condition.json").read_text())
        if trained["arm"] != arm or trained["split_hash"] != SPLIT_HASH:
            raise ValueError("Adapter source condition mismatch")
        # Đối chiếu model hiện tại với condition training trước khi nạp adapter.
        validate_base_identity(trained, MODEL_ID, MODEL_REVISION)
        model, tokenizer, _ = load_student(SEED, adapter_path=adapter)
        condition = {"phase": "final" if EVAL_SPLIT == "test" else "development_fp16_eval32",
                     "card_id": MEASUREMENT_CARD_ID,
                     "measurement_card_sha256": MEASUREMENT_CARD_SHA256,
                     "measurement_card_hash": MEASUREMENT_REGISTRY_SHA256,
                     "source_notebook_output": SOURCE_NOTEBOOK_OUTPUT,
                     "arm": arm, "adapter": str(adapter), "training_condition_hash": canonical_hash(trained),
                     "split_hash": SPLIT_HASH, "metric_id": METRIC_ID, "eval_split": EVAL_SPLIT,
                     "eval_ids": [row["id"] for row in rows], "max_new_tokens": MAX_NEW_TOKENS,
                     "max_length": MAX_LENGTH, "thinking": False}
        # Gắn actual prompt, tokenizer, runtime và weights vào điều kiện inference.
        condition = bind_student_condition(condition, model, tokenizer, rows, adapter)
        if (EVAL_SPLIT == "test" and registration.get("inference_condition_hashes", {}).get(arm)
                != canonical_hash(condition)):
            raise ValueError("Actual final inference condition differs from registration")
        out = WORK_DIR / "runs" / arm / canonical_hash(condition)[:12]
        write_json(out / "condition.json", condition)
        # Hàm dùng chung giữ đúng manifest, resume và mẫu số cho mọi adapter.
        predictions = predict_rows(model, tokenizer, rows, out / "predictions.jsonl", condition)
        report = summarize_predictions(predictions, rows)
        write_json(out / "report.json", report)
        CURRENT_EVAL_RUNS[arm] = {"report": report, "condition": condition, "out": str(out)}
        print({"arm": arm, **{key: value for key, value in report.items() if key != "scores"}})
        release_student(model)
        del model
else:
    print("Adapter evaluation configured; EVAL_ARMS=().")


## So sánh cặp FP16 trên cùng 32 câu

Báo cáo EA/PA/validity và số câu cứu được/làm hỏng thêm. Đây là development smoke, chưa phải kiểm định luận văn.


In [ ]:

import math


def compare_paired_reports(p, g, p_condition, g_condition):
    keys = ("split_hash", "metric_id", "actual_model_id", "actual_model_revision",
            "actual_tokenizer_revision", "inference_prompt_sha256", "inference_seed",
            "max_length", "max_new_tokens", "thinking", "eval_ids", "inference_runtime")
    for key in keys:
        if key not in p_condition or key not in g_condition or p_condition[key] != g_condition[key]:
            raise ValueError("Paired inference condition mismatch: " + key)
    ids = p_condition["eval_ids"]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate paired manifest ID")
    maps = []
    for report in (p, g):
        scores = {row["id"]: row for row in report["scores"]}
        if (report["n"] != len(ids) or len(scores) != len(report["scores"])
                or set(scores) != set(ids) or report["metric_id"] != p_condition["metric_id"]):
            raise ValueError("Paired report IDs/denominator/metric mismatch")
        maps.append(scores)
    if not ids:
        raise ValueError("Empty paired report")
    def aggregate(rows):
        return {key: sum(bool(row[key]) for row in rows.values()) / len(ids)
                for key in ("ea", "pa", "valid")}
    before, after = maps
    return {
        "n": len(ids), "metric_id": p["metric_id"],
        "P": aggregate(before), "G_all": aggregate(after),
        "rescued_ea": sum(not before[key]["ea"] and after[key]["ea"] for key in ids),
        "regressed_ea": sum(before[key]["ea"] and not after[key]["ea"] for key in ids),
        "inference_precision": p_condition["inference_runtime"]["model_dtype"],
        "scope": "development_validation_smoke_not_confirmatory",
    }


In [ ]:

# Chỉ so hai lượt FP16 mới, cùng ID và prompt; không lấy baseline BF16 cũ để thay thế.
if RUN_P and "G_all" in EVAL_ARMS:
    if not all(arm in CURRENT_EVAL_RUNS for arm in ("P", "G_all")):
        raise ValueError("Both current FP16 reports are required")
    p_run, g_run = CURRENT_EVAL_RUNS["P"], CURRENT_EVAL_RUNS["G_all"]
    paired_summary = compare_paired_reports(
        p_run["report"], g_run["report"], p_run["condition"], g_run["condition"])
    paired_summary.update({
        "card_id": MEASUREMENT_CARD_ID, "source_notebook_output": SOURCE_NOTEBOOK_OUTPUT,
        "source_training_condition_hash": SOURCE_TRAINING_CONDITION_HASH,
        "source_adapter_sha256": SOURCE_ADAPTER_SHA256,
        "report_paths": {arm: value["out"] for arm, value in CURRENT_EVAL_RUNS.items()},
    })
    write_json(WORK_DIR / "paired_eval_summary.json", paired_summary)
    print(json.dumps(paired_summary, ensure_ascii=False, indent=2))
else:
    print("Paired summary needs RUN_P=True and EVAL_ARMS=('G_all',).")


## 12. Checklist và artifact

- CPU setup: hashes, source-disjoint split, gold/scorer audit, exports G_all và các arm matched nếu đã có candidates.
- GPU smoke: cài stack, chạy P trên validation, rồi G_all với 20 optimizer step; kiểm tra full checkpoint resume.
- Trước thí nghiệm chính: audit nhãn/nguồn, khóa metric adapter, teacher pool, matched token budget,
  package lock, measurement card và registration theo plan.
- Sau pilot mới bật các lần training đầy đủ. Không chọn kết quả tốt nhất từ test hoặc từ nhiều seed.
- Checkpoint và output nằm dưới /kaggle/working/vinumqa-baselines; lưu version/output trước khi kết thúc session.
  Files chỉ trên working disk không tự tồn tại sau khi session mất.

Nguồn:
[ViNumQA](https://aclanthology.org/2025.vlsp-1.25.pdf),
[data mirror pin](https://github.com/ThanhDatVN/vinumqa-numerical-reasoning/tree/281bdcd70abde7753d95014dbd9babec7f97f3fd),
[FinQA](https://github.com/czyssrs/FinQA),
[Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B),
[PEFT QLoRA](https://huggingface.co/docs/peft/developer_guides/quantization).

In [ ]:
# BIÊN NHẬN SETUP: tóm tắt cấu hình/cờ; bản thân file này không chứng minh GPU đã chạy.
setup_summary = {
    "status": "setup_only_no_model_run" if not (RUN_P or TRAIN_ARMS or EVAL_ARMS) else "development_flags_enabled",
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "card_id": MEASUREMENT_CARD_ID, "compute_precision": COMPUTE_PRECISION,
    "source_notebook_output": SOURCE_NOTEBOOK_OUTPUT,
    "metric_id": METRIC_ID, "split_hash": globals().get("SPLIT_HASH"),
    "counts": globals().get("split_audit", {}).get("counts"),
    "pack_audit": globals().get("pack_audit"),
    "run_p": RUN_P, "train_arms": list(TRAIN_ARMS), "eval_arms": list(EVAL_ARMS),
    "final_test_enabled": ALLOW_FINAL_TEST,
}
# Biên nhận giúp kiểm tra setup và các cờ đã chọn, không chứa điểm model giả lập.
write_json(WORK_DIR / "setup_summary.json", setup_summary)
print(json.dumps(setup_summary, ensure_ascii=False, indent=2))
